# Logistic Regression: Sigmoid, Log Loss and Gradient Descent

This lesson explains how Logistic Regression turns a linear score into a probability, then learns the best weights using **log loss**.

![Logistic regression log loss](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AChMN5rYq5Y_ePchMDwrD-w.png)

Image source: [Logistic Loss Function Explained](https://medium.com/%40mennaafi/classification-with-logistic-regression-and-overfitting-c9681a89b). Formula reference: [scikit-learn log loss documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html).

## 1. Logistic Regression hypothesis

First create a linear score:

    z = theta_0 + theta_1 x_1 + theta_2 x_2 + ...

Then squash it with sigmoid:

    probability = 1 / (1 + exp(-z))

The output is the probability of class 1. Whatever z is, probability stays between 0 and 1.

In [ ]:
# Cell 1: imports and the sigmoid function
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.array([-8, -2, 0, 2, 8])
pd.DataFrame({'z (linear score)': z_values, 'sigmoid(z) = probability': sigmoid(z_values).round(4)})

### Read this edit

The model does not return the raw linear score. It sends that score through sigmoid. A negative score gives probability below 0.5, zero gives 0.5, and a positive score gives probability above 0.5.

In [ ]:
# Cell 2: visualise the sigmoid squash
z_grid = np.linspace(-10, 10, 400)
plt.figure(figsize=(8, 4))
plt.plot(z_grid, sigmoid(z_grid), color='#4C78A8', linewidth=3, label='sigmoid(z)')
plt.axhline(0.5, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.scatter(z_values, sigmoid(z_values), color='#E45756', zorder=3)
plt.xlabel('z: weighted linear score')
plt.ylabel('Probability of class 1')
plt.title('Sigmoid squashes every score into 0 to 1')
plt.legend()
plt.show()

## 2. Why use log loss?

Putting sigmoid predictions inside the Linear Regression squared-error cost can create a non-convex optimization surface. Logistic Regression instead uses log loss, also called binary cross-entropy.

For one example:

    loss = -[y log(probability) + (1 - y) log(1 - probability)]

When y is 1 this becomes -log(probability). When y is 0 it becomes -log(1 - probability).

In [ ]:
# Cell 3: plot log loss for both true labels
probability_grid = np.linspace(0.001, 0.999, 400)
loss_when_y1 = -np.log(probability_grid)
loss_when_y0 = -np.log(1 - probability_grid)
plt.figure(figsize=(8, 4))
plt.plot(probability_grid, loss_when_y1, label='True label = 1', color='#54A24B', linewidth=3)
plt.plot(probability_grid, loss_when_y0, label='True label = 0', color='#E45756', linewidth=3)
plt.ylim(0, 8)
plt.xlabel('Predicted probability of class 1')
plt.ylabel('Log loss')
plt.title('Confident wrong probabilities get a large penalty')
plt.legend()
plt.show()

### How to read the loss diagram

- For a true label of 1, probability near 1 gives loss near 0.
- For a true label of 0, probability near 0 gives loss near 0.
- A confident wrong prediction gets a very large penalty. This teaches the model to give sensible probabilities.

In [ ]:
# Cell 4: calculate log loss manually and with scikit-learn
true_labels = np.array([1, 0, 1, 0])
predicted_probabilities = np.array([0.90, 0.20, 0.70, 0.10])
manual_loss = -np.mean(true_labels * np.log(predicted_probabilities) + (1 - true_labels) * np.log(1 - predicted_probabilities))
library_loss = log_loss(true_labels, predicted_probabilities)
print('Manual mean log loss:', round(manual_loss, 4))
print('scikit-learn log loss:', round(library_loss, 4))

## 3. Cost for all training examples

For m examples, average all individual losses:

    J(theta) = -(1/m) sum [y log(probability) + (1-y) log(1-probability)]

The goal is to find theta values that make J(theta) as small as possible. The log-loss objective for standard logistic regression is convex, so it has one global minimum.

In [ ]:
# Cell 5: visualise a small logistic-regression cost surface
x_small = np.array([-2, -1, 1, 2])
y_small = np.array([0, 0, 1, 1])
theta0_grid, theta1_grid = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-4, 4, 100))
z = theta0_grid[..., None] + theta1_grid[..., None] * x_small
probability = np.clip(sigmoid(z), 1e-10, 1 - 1e-10)
cost = -np.mean(y_small * np.log(probability) + (1 - y_small) * np.log(1 - probability), axis=2)
plt.figure(figsize=(7, 5))
contours = plt.contourf(theta0_grid, theta1_grid, cost, levels=25, cmap='viridis')
plt.colorbar(contours, label='Average log loss')
plt.xlabel('theta_0: intercept')
plt.ylabel('theta_1: feature weight')
plt.title('Log-loss surface: move toward the lowest point')
plt.show()

## 4. Gradient descent in one idea

Gradient descent repeats this update until cost stops improving:

    theta_j = theta_j - learning_rate × slope

The learning rate controls step size. Too large can skip a good solution; too small learns slowly. Libraries choose an optimizer automatically, but this is the core idea.

In [ ]:
# Cell 6: fit Logistic Regression and inspect learned probabilities
model = LogisticRegression(C=1e6, solver='lbfgs').fit(x_small.reshape(-1, 1), y_small)
test_points = np.array([-1.5, 0, 1.5]).reshape(-1, 1)
results = pd.DataFrame({
    'x': test_points.ravel(),
    'Probability of class 1': model.predict_proba(test_points)[:, 1].round(3),
    'Predicted class': model.predict(test_points)
})
results

## Quick revision card

1. Logistic Regression first creates a weighted score z.
2. Sigmoid converts z into probability between 0 and 1.
3. Probability at or above 0.5 commonly becomes class 1; the threshold can change.
4. Log loss rewards correct confident probabilities and strongly penalizes confident wrong ones.
5. The average log-loss cost is minimized by changing the theta weights.
6. Gradient descent repeatedly takes steps that lower cost.

**One-line interview answer:** Logistic Regression applies sigmoid to a linear score and learns its weights by minimizing binary cross-entropy, also called log loss.